# Phase 3 — Final Data Cleaning

## Overview

This notebook performs the final cleaning and validation of the Zillow property dataset before it is used for **feature engineering, exploratory data analysis (EDA), and machine learning**.

The goal is to produce a clean and consistent dataset while preserving meaningful information about missing property and financial attributes.

### Cleaning Objectives

1. **Remove invalid records**
   - Remove properties with `listing_price <= 0`.
   - Remove properties with `living_area <= 0`.
   - Compare row counts before and after filtering.

2. **Correct property-type inconsistencies**
   - Identify listings whose addresses contain strong multi-unit indicators.
   - Correct inconsistent `home_type` values to `MULTI_FAMILY`.
   - Review affected records before and after the correction.

3. **Handle missing values**
   - Create missing-value indicators for `zestimate`, `rent_estimate`, and `tax_assessed_value`.
   - Impute missing `bedrooms`, `bathrooms`, and `living_area` values using their respective medians.
   - Preserve missing financial values rather than introducing artificial estimates.

4. **Validate and export the final dataset**
   - Check for remaining invalid values and duplicate rows.
   - Review the final home-type distribution.
   - Export the cleaned dataset as `final_cleaned_zillow.csv`.

### Input and Output

**Input:** `final2_df.csv` — dataset produced during the previous cleaning phase.

**Output:** `final_cleaned_zillow.csv` — final Zillow dataset prepared for the next stage of the NYC real estate price prediction project.


## 1. Load the Dataset

Load the dataset produced in the previous cleaning phase and inspect its initial dimensions.

> **Note:** The input file should be located in the notebook's working directory. If necessary, update `INPUT_FILE` to the correct local path.


In [26]:
import pandas as pd
import numpy as np

# Define the input file generated during the previous cleaning phase.
INPUT_FILE = "final2_df.csv"

# Load the Zillow dataset.
df = pd.read_csv(INPUT_FILE)

print(f"Initial dataset shape: {df.shape}")
df.head()


Initial dataset shape: (1892, 23)


,address,zip_code,bedrooms,bathrooms,living_area,latitude,longitude,listing_price,rent_estimate,zestimate,...,zpid,tax_assessed_value,clean_address,property_key,zestimate_missing,tax_assessed_missing,price_per_sqft,price_outlier,price_vs_zestimate,price_zestimate_ratio
0,"572 Louisiana Avenue #1, Brooklyn, NY 11239",11239,2.0,2.0,1017.0,40.647620,-73.886810,250000.0,3550.0,856400.0,...,30775866,570294.0,572 LOUISIANA AVENUE BROOKLYN NY 11239,30775866,1,0,245.821042,0,-606400.0,0.291920
1,"2183 Ocean Ave #4A, Brooklyn, NY 11229",11229,1.0,1.0,541.0,40.608430,-73.952736,485000.0,2489.0,473800.0,...,463914912,817000.0,2183 OCEAN AVE BROOKLYN NY 11229,463914912,0,1,896.487985,0,11200.0,1.023639
2,"8885 15th Ave, Brooklyn, NY 11228",11228,4.0,3.0,2060.0,40.605550,-74.015200,1299000.0,5015.0,1315600.0,...,30714518,1320000.0,8885 15TH AVE BROOKLYN NY 11228,30714518,0,0,630.582524,0,-16600.0,0.987382
3,"300 Milford St, Brooklyn, NY 11208",11208,3.0,2.0,1232.0,40.671783,-73.876625,595000.0,3574.0,602800.0,...,30640361,729000.0,300 MILFORD ST BROOKLYN NY 11208,30640361,0,0,482.954545,0,-7800.0,0.987060
4,"2711 Avenue X Unit F2, Brooklyn, NY 11235",11235,2.0,1.0,1000.0,40.704254,-73.966420,345000.0,3056.0,350500.0,...,463826019,817000.0,2711 AVENUE X UNIT F2 BROOKLYN NY 11235,463826019,0,1,345.000000,0,-5500.0,0.984308


## 2. Remove Invalid Records

Property records with a non-positive listing price or living area cannot be used reliably for price prediction.

The following validation rules are applied:

- `listing_price > 0`
- `living_area > 0`

The number of affected records is measured before filtering so the cleaning impact can be documented.


In [27]:
# Record the number of rows before applying validation rules.
rows_before = len(df)

# Count records that violate the required price and living-area rules.
invalid_listing_price = (df["listing_price"] <= 0).sum()
invalid_living_area = (df["living_area"] <= 0).sum()

print(f"Rows before cleaning: {rows_before:,}")
print(f"Invalid listing_price records: {invalid_listing_price:,}")
print(f"Invalid living_area records: {invalid_living_area:,}")


Rows before cleaning: 1,892
Invalid listing_price records: 0
Invalid living_area records: 0


In [28]:
# Keep only records with valid positive listing prices and living areas.
df = df[
    (df["listing_price"] > 0) &
    (df["living_area"] > 0)
].copy()

rows_after = len(df)
rows_removed = rows_before - rows_after

print(f"Rows after cleaning: {rows_after:,}")
print(f"Rows removed: {rows_removed:,}")


Rows after cleaning: 1,892
Rows removed: 0


In [29]:
# Verify that no invalid price or living-area records remain.
remaining_invalid_price = (df["listing_price"] <= 0).sum()
remaining_invalid_area = (df["living_area"] <= 0).sum()

print(f"Invalid listing_price records remaining: {remaining_invalid_price:,}")
print(f"Invalid living_area records remaining: {remaining_invalid_area:,}")


Invalid listing_price records remaining: 0
Invalid living_area records remaining: 0


In [30]:
# Save the dataset after removing invalid records.
STEP_1_FILE = "cleaning_step_1.csv"

df.to_csv(STEP_1_FILE, index=False)

print(f"Saved intermediate dataset: {STEP_1_FILE}")


Saved intermediate dataset: cleaning_step_1.csv


## 3. Correct Home-Type Inconsistencies

Some listings may be classified as `SINGLE_FAMILY` even though the address contains a strong indicator that the property is multi-unit.

For example:

`636 E 96th St #6UNITS`

should be treated as a multi-family property rather than a single-family property.

The address is used only to identify **strong multi-unit indicators**. Matching records are reviewed before the `home_type` value is changed.


In [31]:
# Review the existing distribution of property types before correction.
print("Home type distribution before correction:")
print(df["home_type"].value_counts(dropna=False))


Home type distribution before correction:
home_type
SINGLE_FAMILY    707
MULTI_FAMILY     576
CONDO            565
LOT               34
MANUFACTURED       5
TOWNHOUSE          3
APARTMENT          2
Name: count, dtype: int64


In [32]:
# Define strong address patterns that indicate a multi-unit property.
multi_unit_pattern = r"\b\d+\s*UNITS?\b|\b\d+FAM\b|\bMULTI[-\s]?FAMILY\b"

# Normalize addresses before applying the pattern.
address_text = df["address"].fillna("").astype(str).str.upper()

# Identify listings whose addresses contain a strong multi-unit indicator.
multi_unit_mask = address_text.str.contains(
    multi_unit_pattern,
    regex=True,
    na=False
)

print(f"Potential multi-unit records identified: {multi_unit_mask.sum():,}")


Potential multi-unit records identified: 7


In [33]:
# Review the records that will be reclassified before making the change.
df.loc[
    multi_unit_mask,
    ["address", "home_type"]
].head(20)


,address,home_type
423,"448 E 48th St #6FAM, Brooklyn, NY 11203",MULTI_FAMILY
500,"454 E 48th St #6FAM, Brooklyn, NY 11203",MULTI_FAMILY
693,"636 E 96th St #6UNITS, Brooklyn, NY 11236",MULTI_FAMILY
739,"341 E 51st St #2FAM, Brooklyn, NY 11203",MULTI_FAMILY
901,"4543 Beach 46th St Unit 1FAM, Brooklyn, NY 11224",MULTI_FAMILY
1166,"98-19 160th Ave #2FAM, Howard Beach, NY 11414",MULTI_FAMILY
1652,"145-11 167th St #2FAM, Jamaica, NY 11434",MULTI_FAMILY


In [34]:
# Reclassify identified multi-unit listings.
df.loc[multi_unit_mask, "home_type"] = "MULTI_FAMILY"

print("Updated home types for identified multi-unit records:")
print(df.loc[multi_unit_mask, "home_type"].value_counts(dropna=False))


Updated home types for identified multi-unit records:
home_type
MULTI_FAMILY    7
Name: count, dtype: int64


In [35]:
# Verify the corrected records.
df.loc[
    multi_unit_mask,
    ["address", "home_type"]
].head(20)


,address,home_type
423,"448 E 48th St #6FAM, Brooklyn, NY 11203",MULTI_FAMILY
500,"454 E 48th St #6FAM, Brooklyn, NY 11203",MULTI_FAMILY
693,"636 E 96th St #6UNITS, Brooklyn, NY 11236",MULTI_FAMILY
739,"341 E 51st St #2FAM, Brooklyn, NY 11203",MULTI_FAMILY
901,"4543 Beach 46th St Unit 1FAM, Brooklyn, NY 11224",MULTI_FAMILY
1166,"98-19 160th Ave #2FAM, Howard Beach, NY 11414",MULTI_FAMILY
1652,"145-11 167th St #2FAM, Jamaica, NY 11434",MULTI_FAMILY


In [36]:
# Review the overall home-type distribution after correction.
print("Home type distribution after correction:")
print(df["home_type"].value_counts(dropna=False))


Home type distribution after correction:
home_type
SINGLE_FAMILY    707
MULTI_FAMILY     576
CONDO            565
LOT               34
MANUFACTURED       5
TOWNHOUSE          3
APARTMENT          2
Name: count, dtype: int64


In [37]:
# Save the dataset after correcting home-type inconsistencies.
STEP_2_FILE = "cleaning_step_2.csv"

df.to_csv(STEP_2_FILE, index=False)

print(f"Saved intermediate dataset: {STEP_2_FILE}")


Saved intermediate dataset: cleaning_step_2.csv


## 4. Handle Missing Values

Missing values are treated differently depending on the role of the feature.

### Property characteristics

Missing values in the following fields are filled with their respective medians:

- `bedrooms`
- `bathrooms`
- `living_area`

Median imputation is used because it is less sensitive to extreme property values than mean imputation.

### Financial attributes

The following fields are intentionally **not imputed**:

- `zestimate`
- `rent_estimate`
- `tax_assessed_value`

Instead, missing-value indicator columns are created. This preserves the information that the original source did not provide a value for the property.


In [38]:
# Define the columns whose missingness will be reviewed.
columns_to_check = [
    "bedrooms",
    "bathrooms",
    "living_area",
    "zestimate",
    "rent_estimate",
    "tax_assessed_value"
]

# Review missing-value counts before treatment.
missing_counts = df[columns_to_check].isna().sum()

print("Missing values before treatment:")
print(missing_counts)


Missing values before treatment:
bedrooms              0
bathrooms             0
living_area           0
zestimate             0
rent_estimate         0
tax_assessed_value    0
dtype: int64


In [39]:
# Calculate the percentage of missing values for each selected column.
missing_summary = pd.DataFrame({
    "missing_count": df[columns_to_check].isna().sum(),
    "missing_percentage": (
        df[columns_to_check].isna().mean() * 100
    ).round(2)
})

missing_summary


,missing_count,missing_percentage
bedrooms,0,0.0
bathrooms,0,0.0
living_area,0,0.0
zestimate,0,0.0
rent_estimate,0,0.0
tax_assessed_value,0,0.0


In [40]:
# Create binary indicators that preserve whether each financial value was missing.
df["zestimate_missing"] = df["zestimate"].isna().astype(int)
df["rent_missing"] = df["rent_estimate"].isna().astype(int)
df["tax_missing"] = df["tax_assessed_value"].isna().astype(int)

# Review the newly created missingness indicators.
df[
    [
        "zestimate",
        "zestimate_missing",
        "rent_estimate",
        "rent_missing",
        "tax_assessed_value",
        "tax_missing"
    ]
].head(10)


,zestimate,zestimate_missing,rent_estimate,rent_missing,tax_assessed_value,tax_missing
0,856400.0,0,3550.0,0,570294.0,0
1,473800.0,0,2489.0,0,817000.0,0
2,1315600.0,0,5015.0,0,1320000.0,0
3,602800.0,0,3574.0,0,729000.0,0
4,350500.0,0,3056.0,0,817000.0,0
5,599700.0,0,3050.0,0,570000.0,0
6,938000.0,0,3929.0,0,804000.0,0
7,856400.0,0,3467.0,0,817000.0,0
8,856400.0,0,4245.0,0,742000.0,0
9,3424700.0,0,2705.0,0,1361000.0,0


In [41]:
# Calculate the median for each property characteristic.
median_values = df[
    ["bedrooms", "bathrooms", "living_area"]
].median()

print("Median values used for imputation:")
print(median_values)


Median values used for imputation:
bedrooms          3.0
bathrooms         2.0
living_area    1467.0
dtype: float64


In [42]:
# Fill missing property characteristics using their respective median values.
for column, median_value in median_values.items():
    df[column] = df[column].fillna(median_value)


In [43]:
# Verify that the selected property characteristics no longer contain missing values.
property_columns = ["bedrooms", "bathrooms", "living_area"]

print("Missing property characteristics after imputation:")
print(df[property_columns].isna().sum())


Missing property characteristics after imputation:
bedrooms       0
bathrooms      0
living_area    0
dtype: int64


In [44]:
# Confirm that financial fields remain missing where the original data was unavailable.
financial_columns = [
    "zestimate",
    "rent_estimate",
    "tax_assessed_value"
]

print("Remaining missing financial values:")
print(df[financial_columns].isna().sum())


Remaining missing financial values:
zestimate             0
rent_estimate         0
tax_assessed_value    0
dtype: int64


## 5. Final Dataset Validation

Before exporting the final dataset, perform a final quality check to confirm that the cleaning rules were applied successfully.

The validation includes:

- Final dataset dimensions
- Duplicate-row count
- Invalid listing prices
- Invalid living areas
- Remaining missing values
- Final home-type distribution


In [45]:
# Display the final dataset dimensions.
print(f"Final dataset shape: {df.shape}")


Final dataset shape: (1892, 25)


In [46]:
# Check for completely duplicated rows.
duplicate_rows = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_rows:,}")


Duplicate rows: 0


In [47]:
# Confirm that no non-positive listing prices remain.
print(
    "Invalid listing_price records:",
    (df["listing_price"] <= 0).sum()
)

# Confirm that no non-positive living areas remain.
print(
    "Invalid living_area records:",
    (df["living_area"] <= 0).sum()
)


Invalid listing_price records: 0
Invalid living_area records: 0


In [48]:
# Review all remaining missing values in the final dataset.
final_missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (
        df.isna().mean() * 100
    ).round(2)
})

final_missing_summary[
    final_missing_summary["missing_count"] > 0
].sort_values(
    "missing_count",
    ascending=False
)


,missing_count,missing_percentage


In [49]:
# Review the final distribution of property types.
print("Final home type distribution:")
print(df["home_type"].value_counts(dropna=False))


Final home type distribution:
home_type
SINGLE_FAMILY    707
MULTI_FAMILY     576
CONDO            565
LOT               34
MANUFACTURED       5
TOWNHOUSE          3
APARTMENT          2
Name: count, dtype: int64


## 6. Export Final Cleaned Dataset

The cleaned and validated Zillow dataset is exported for use in the next stages of the project, including feature engineering, exploratory analysis, and model development.


In [50]:
# Export the final cleaned Zillow dataset.
OUTPUT_FILE = "final_cleaned_zillow.csv"

df.to_csv(OUTPUT_FILE, index=False)

print(f"Final dataset saved successfully: {OUTPUT_FILE}")
print(f"Final dataset shape: {df.shape}")


Final dataset saved successfully: final_cleaned_zillow.csv
Final dataset shape: (1892, 25)


## Conclusion

The final cleaning phase has prepared the Zillow dataset for downstream analysis and modeling.

The dataset has been cleaned by:

- Removing records with invalid `listing_price` or `living_area` values.
- Correcting selected multi-unit property classifications based on strong address indicators.
- Creating missing-value indicators for Zillow financial attributes.
- Imputing missing `bedrooms`, `bathrooms`, and `living_area` values using median values.
- Preserving unavailable `zestimate`, `rent_estimate`, and `tax_assessed_value` values as missing.
- Performing final validation checks for invalid records, duplicates, missing values, and property-type consistency.

The resulting dataset is saved as:

`final_cleaned_zillow.csv`

This file will be used as the cleaned input for the **feature engineering and modeling phase** of the NYC real estate price prediction project.
